## Introduction

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Import from our custom src package
from src import (
    TweetDataPreprocessor,
    TweetDataset,
    StackedBiLSTMAttention,
    ModelTrainer,
    ModelEvaluator
)

preprocessor = TweetDataPreprocessor('preprocessed_tweets.csv')
X, y = preprocessor.process_data()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

train_dataset = TweetDataset(X_train, y_train)
val_dataset = TweetDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

#### Data preparation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = StackedBiLSTMAttention(
    vocab_size=preprocessor.vocab_size,
    embed_dim=100,
    hidden_dim=64,
    output_dim=3,
    num_layers=2,
    dropout=0.3
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

#### Model initialization

In [ ]:
evaluator = ModelEvaluator(criterion, device)
trainer = ModelTrainer(model, optimizer, criterion, device)

trainer.train(epochs=5, train_loader=train_loader, val_loader=val_loader, evaluator=evaluator)

#### Training

In [ ]:
trainer.plot_learning_curves(trainer.train_losses, trainer.val_losses)

_, y_true, y_pred = evaluator.evaluate(model, val_loader)
evaluator.plot_confusion_matrix(y_true, y_pred)

#### Evaluation & Plots